## Category important words & similarity search

In [101]:
import pandas as pd
import numpy as np
import re
import time
import nltk
#from nltk import bigrams, trigrams
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity


#model = SentenceTransformer('/Users/zphilipp/git/research/relevance/models/sentence-transformer.model')
model = SentenceTransformer('all-MiniLM-L6-v2')

#pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 200)

prepositions_and_conjunctions = [
    "about", "above", "across", "after", "against", "along", "among", "around", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "to", "toward", "under",
    "until", "up", "with", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a"
]
pattern = r'\b(?:' + '|'.join(prepositions_and_conjunctions) + r')\b'

def remove_prepositions_and_conjunctions(text):
    text = text.lower()
    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text.replace("-", "")

/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [104]:
df = pd.read_csv('data/top_category_name.csv')
df

,Unnamed: 0,id,name,path
0,0,1bf81df3-efc0-44f3-937f-8d19e12e1530,Nightlife,nearby things to do nightlife
1,1,2afb5ba6-75bc-497e-a53a-be118d00f59a,Sightseeing & Tours,nearby things to do sightseeing & tours
2,2,634fe6f0-7ac5-4ca0-96d5-7f63f97f9cd2,Bus Tours & Rentals,nearby things to do sightseeing & tours bus tours & rentals
3,3,8fce0ab5-5a6a-480d-ac35-f1586b66f857,Brows & Lashes,nearby beauty & spas brows & lashes
4,4,91e47ef0-776d-4eaf-9acb-acd360036b94,Eyelash Extensions,nearby beauty & spas brows & lashes eyelash extensions
...,...,...,...,...
89,89,93405d8f-cbd5-456e-ad46-68133fd0332f,Wine,goods grocery & household alcohol wine
90,90,0bc79434-fc4c-482a-b19d-e8af6cc0ac5b,Music,nearby things to do tickets & events music
91,91,e523df1e-6f08-42dd-92b1-2d16650eaf2f,Golf,nearby things to do sports & outdoors golf
92,92,5b09094c-d168-4f38-b05e-bb620f45d363,Plastic Surgery,nearby health & fitness medical plastic surgery


In [105]:
df['text'] = df['path'].apply(remove_prepositions_and_conjunctions)
df_ = df
df_.head()

,Unnamed: 0,id,name,path,text
0,0,1bf81df3-efc0-44f3-937f-8d19e12e1530,Nightlife,nearby things to do nightlife,nearby things do nightlife
1,1,2afb5ba6-75bc-497e-a53a-be118d00f59a,Sightseeing & Tours,nearby things to do sightseeing & tours,nearby things do sightseeing & tours
2,2,634fe6f0-7ac5-4ca0-96d5-7f63f97f9cd2,Bus Tours & Rentals,nearby things to do sightseeing & tours bus tours & rentals,nearby things do sightseeing & tours bus tours & rentals
3,3,8fce0ab5-5a6a-480d-ac35-f1586b66f857,Brows & Lashes,nearby beauty & spas brows & lashes,nearby beauty & spas brows & lashes
4,4,91e47ef0-776d-4eaf-9acb-acd360036b94,Eyelash Extensions,nearby beauty & spas brows & lashes eyelash extensions,nearby beauty & spas brows & lashes eyelash extensions


#### Get all titles from Deals and Options text

#### Create word embedings and transform data

In [106]:
df_['text'] = df_['text'].tolist()
df_['text_embeddings'] = df_['text'].apply(lambda x: model.encode(x))

combined_embeddings = np.array(df_['text_embeddings'].tolist())

In [109]:
df_.head()

,Unnamed: 0,id,name,path,text,text_embeddings
0,0,1bf81df3-efc0-44f3-937f-8d19e12e1530,Nightlife,nearby things to do nightlife,nearby things do nightlife,"[0.10719261, -0.009873317, -0.002886109, 0.04016389, 0.02574348, -0.055390447, 0.04755237, -0.09206853, 0.041121837, -0.024387462, 0.060728706, -0.048695356, -0.0129105635, 0.0455808, 0.107071236,..."
1,1,2afb5ba6-75bc-497e-a53a-be118d00f59a,Sightseeing & Tours,nearby things to do sightseeing & tours,nearby things do sightseeing & tours,"[0.110048324, -0.03330038, 0.05407769, 0.00047145097, 0.025682904, -0.054179642, 0.056862134, -0.032360036, -0.055944394, 0.0068316287, 0.0582175, 0.0161768, -0.010869576, 0.10275831, 0.07755927, ..."
2,2,634fe6f0-7ac5-4ca0-96d5-7f63f97f9cd2,Bus Tours & Rentals,nearby things to do sightseeing & tours bus tours & rentals,nearby things do sightseeing & tours bus tours & rentals,"[0.10644985, -0.04905953, 0.054852676, 0.016697181, -0.035579592, 0.006469342, 0.075778306, -0.025403628, -0.022825733, -0.024200391, 0.04719262, 0.053419624, 0.02457347, 0.09207393, 0.08980535, -..."
3,3,8fce0ab5-5a6a-480d-ac35-f1586b66f857,Brows & Lashes,nearby beauty & spas brows & lashes,nearby beauty & spas brows & lashes,"[0.010597926, -0.052663118, 0.064909294, 0.060403436, -0.049161762, -0.013531411, 0.0118945055, -0.06260367, -0.1079998, -0.028962698, 0.12617949, -0.0657222, -0.046697628, 0.047157846, 0.04524961..."
4,4,91e47ef0-776d-4eaf-9acb-acd360036b94,Eyelash Extensions,nearby beauty & spas brows & lashes eyelash extensions,nearby beauty & spas brows & lashes eyelash extensions,"[-0.010798341, -0.038259745, 0.0706647, 0.06036469, -0.017027609, -0.033844322, 0.0064394977, -0.033618566, -0.09275439, -0.037446145, 0.15190047, -0.03374619, -0.03807696, 0.041630223, 0.06628009..."


In [142]:
def query_embedding_reduce(query_embedding):
    if query_embedding.shape[1] > 384:
        query_embedding_reduced = np.mean(query_embedding.reshape(-1, 2, 384), axis=1)
    else:
        query_embedding_reduced = query_embedding
    return query_embedding_reduced

df_[['id', 'name', 'text', 'text_embeddings']].to_csv('models/category_embeding_top.csv')
df_.head(5)

,Unnamed: 0,id,name,path,text,text_embeddings
0,0,1bf81df3-efc0-44f3-937f-8d19e12e1530,Nightlife,nearby things to do nightlife,nearby things do nightlife,"[0.10719261, -0.009873317, -0.002886109, 0.04016389, 0.02574348, -0.055390447, 0.04755237, -0.09206853, 0.041121837, -0.024387462, 0.060728706, -0.048695356, -0.0129105635, 0.0455808, 0.107071236,..."
1,1,2afb5ba6-75bc-497e-a53a-be118d00f59a,Sightseeing & Tours,nearby things to do sightseeing & tours,nearby things do sightseeing & tours,"[0.110048324, -0.03330038, 0.05407769, 0.00047145097, 0.025682904, -0.054179642, 0.056862134, -0.032360036, -0.055944394, 0.0068316287, 0.0582175, 0.0161768, -0.010869576, 0.10275831, 0.07755927, ..."
2,2,634fe6f0-7ac5-4ca0-96d5-7f63f97f9cd2,Bus Tours & Rentals,nearby things to do sightseeing & tours bus tours & rentals,nearby things do sightseeing & tours bus tours & rentals,"[0.10644985, -0.04905953, 0.054852676, 0.016697181, -0.035579592, 0.006469342, 0.075778306, -0.025403628, -0.022825733, -0.024200391, 0.04719262, 0.053419624, 0.02457347, 0.09207393, 0.08980535, -..."
3,3,8fce0ab5-5a6a-480d-ac35-f1586b66f857,Brows & Lashes,nearby beauty & spas brows & lashes,nearby beauty & spas brows & lashes,"[0.010597926, -0.052663118, 0.064909294, 0.060403436, -0.049161762, -0.013531411, 0.0118945055, -0.06260367, -0.1079998, -0.028962698, 0.12617949, -0.0657222, -0.046697628, 0.047157846, 0.04524961..."
4,4,91e47ef0-776d-4eaf-9acb-acd360036b94,Eyelash Extensions,nearby beauty & spas brows & lashes eyelash extensions,nearby beauty & spas brows & lashes eyelash extensions,"[-0.010798341, -0.038259745, 0.0706647, 0.06036469, -0.017027609, -0.033844322, 0.0064394977, -0.033618566, -0.09275439, -0.037446145, 0.15190047, -0.03374619, -0.03807696, 0.041630223, 0.06628009..."


In [122]:
def get_top_similarity(query_embedding_reduced, combined_embeddings):
    similarities = cosine_similarity(query_embedding_reduced, combined_embeddings).flatten()
    closest_indices = np.argsort(similarities)[-10:]

    closest_rows = []
    for index in reversed(closest_indices):
        if similarities[index] > 0.3:
        
            closest_rows.append([df_.iloc[index], similarities[index]])

    return closest_rows

### Test query -> category use Cosine similarity of category embedings and query embedings

In [123]:
def get_sim(query):
    start_time = time.time()
    query_embedding_reduced = query_embedding_reduce(model.encode(query).reshape(1, -1))
    print (f"Embeding time :{time.time() - start_time}")
    result = get_top_similarity(query_embedding_reduced, combined_embeddings)
    print (f"Total run time :{time.time() - start_time}")
    for row in result:
        print(f"Closest Category: <{row[0]['name']}> -> score {row[1]}")

In [124]:
get_sim(['massage', 'oil'])

Embeding time :0.1533949375152588
Total run time :0.1572113037109375
Closest Category: <Massage> -> score 0.5936518907546997
Closest Category: <Massage> -> score 0.5578334331512451
Closest Category: <Massage> -> score 0.545110821723938
Closest Category: <Deep Tissue Massage> -> score 0.5422124266624451
Closest Category: <Full Body Massage> -> score 0.5116969347000122
Closest Category: <Custom Massage> -> score 0.5020962953567505
Closest Category: <Reflexology> -> score 0.49334287643432617
Closest Category: <Swedish Massage> -> score 0.48978015780448914
Closest Category: <Hot Stone Massage> -> score 0.47802236676216125
Closest Category: <Couples Massage> -> score 0.4769304394721985


In [125]:
get_sim(['oil'])

Embeding time :0.02761387825012207
Total run time :0.03037095069885254
Closest Category: <Oil Change> -> score 0.42151808738708496


In [126]:
get_sim(['change'])

Embeding time :0.06753778457641602
Total run time :0.06848001480102539


In [127]:
get_sim(['oil', 'change'])

Embeding time :0.05665087699890137
Total run time :0.057589054107666016
Closest Category: <Oil Change> -> score 0.37282800674438477


In [128]:
get_sim(['sauna', 'massage'])

Embeding time :0.46227121353149414
Total run time :0.4723949432373047
Closest Category: <Deep Tissue Massage> -> score 0.6411435008049011
Closest Category: <Full Body Massage> -> score 0.6267561912536621
Closest Category: <Massage> -> score 0.6256269216537476
Closest Category: <Reflexology> -> score 0.6165924072265625
Closest Category: <Custom Massage> -> score 0.5982100963592529
Closest Category: <Massage> -> score 0.5961123704910278
Closest Category: <Saunas> -> score 0.5924926996231079
Closest Category: <Hot Stone Massage> -> score 0.5724921226501465
Closest Category: <Swedish Massage> -> score 0.5599960684776306
Closest Category: <Massage> -> score 0.5589667558670044


In [129]:
get_sim(['oil', 'massage'])

Embeding time :0.06228280067443848
Total run time :0.07206463813781738
Closest Category: <Massage> -> score 0.5936518907546997
Closest Category: <Massage> -> score 0.5578334331512451
Closest Category: <Massage> -> score 0.545110821723938
Closest Category: <Deep Tissue Massage> -> score 0.5422124266624451
Closest Category: <Full Body Massage> -> score 0.5116969347000122
Closest Category: <Custom Massage> -> score 0.5020962953567505
Closest Category: <Reflexology> -> score 0.49334287643432617
Closest Category: <Swedish Massage> -> score 0.48978015780448914
Closest Category: <Hot Stone Massage> -> score 0.47802236676216125
Closest Category: <Couples Massage> -> score 0.4769304394721985


In [130]:
get_sim(['massage', 'oil'])

Embeding time :0.13997626304626465
Total run time :0.1409142017364502
Closest Category: <Massage> -> score 0.5936518907546997
Closest Category: <Massage> -> score 0.5578334331512451
Closest Category: <Massage> -> score 0.545110821723938
Closest Category: <Deep Tissue Massage> -> score 0.5422124266624451
Closest Category: <Full Body Massage> -> score 0.5116969347000122
Closest Category: <Custom Massage> -> score 0.5020962953567505
Closest Category: <Reflexology> -> score 0.49334287643432617
Closest Category: <Swedish Massage> -> score 0.48978015780448914
Closest Category: <Hot Stone Massage> -> score 0.47802236676216125
Closest Category: <Couples Massage> -> score 0.4769304394721985


In [131]:
get_sim(['valvoline', 'oil'])

Embeding time :0.07373690605163574
Total run time :0.08414697647094727
Closest Category: <Oil Change> -> score 0.3273688554763794
Closest Category: <Massage> -> score 0.3056454360485077


In [132]:
get_sim(['water'])

Embeding time :0.0834507942199707
Total run time :0.08791184425354004
Closest Category: <Water Sports> -> score 0.36752113699913025
Closest Category: <Colonic Hydrotherapy> -> score 0.31639203429222107


In [133]:
get_sim(['water', 'parks'])

Embeding time :0.05859994888305664
Total run time :0.06261682510375977
Closest Category: <Water Sports> -> score 0.4425596594810486
Closest Category: <Sports & Outdoors> -> score 0.3915403485298157
Closest Category: <Sports & Outdoors> -> score 0.35566163063049316
Closest Category: <Fun & Leisure> -> score 0.3468207120895386
Closest Category: <Yoga> -> score 0.33923429250717163
Closest Category: <Golf> -> score 0.31867673993110657
Closest Category: <Amusement Parks> -> score 0.31864893436431885
Closest Category: <Escape Games> -> score 0.31035852432250977
Closest Category: <Wine> -> score 0.3075743317604065
Closest Category: <Golf> -> score 0.30693745613098145


In [134]:
get_sim(['amc'])

Embeding time :0.06035304069519043
Total run time :0.07455587387084961


In [135]:
get_sim(['pilates'])

Embeding time :0.03254818916320801
Total run time :0.03772711753845215
Closest Category: <Pilates> -> score 0.6689449548721313
Closest Category: <Plastic Surgery> -> score 0.4212318956851959
Closest Category: <Yoga> -> score 0.368234783411026
Closest Category: <Chiropractor> -> score 0.354994535446167
Closest Category: <Medical> -> score 0.352630615234375
Closest Category: <Acupuncture> -> score 0.31720924377441406


In [136]:
get_sim(['ring'])

Embeding time :0.03050994873046875
Total run time :0.041821956634521484


In [137]:
get_sim(['wheel'])

Embeding time :0.060529232025146484
Total run time :0.06162309646606445
Closest Category: <Tires & Wheels> -> score 0.44538387656211853


In [138]:
get_sim(['nail'])

Embeding time :0.059094905853271484
Total run time :0.06505870819091797
Closest Category: <Mani Pedi> -> score 0.376738965511322


In [139]:
get_sim(['massage', 'palace'])

Embeding time :0.06759405136108398
Total run time :0.0728151798248291
Closest Category: <Massage> -> score 0.6104089021682739
Closest Category: <Full Body Massage> -> score 0.6042941808700562
Closest Category: <Deep Tissue Massage> -> score 0.5990667343139648
Closest Category: <Custom Massage> -> score 0.5824674367904663
Closest Category: <Reflexology> -> score 0.5727773904800415
Closest Category: <Hot Stone Massage> -> score 0.5465376377105713
Closest Category: <Massage> -> score 0.542343258857727
Closest Category: <Couples Massage> -> score 0.5377268195152283
Closest Category: <Swedish Massage> -> score 0.5308575630187988
Closest Category: <Massage> -> score 0.5209032893180847


In [140]:
get_sim(['apple', 'cider'])

Embeding time :0.032115936279296875
Total run time :0.03850221633911133
Closest Category: <Electronics> -> score 0.41071468591690063
Closest Category: <Electronics> -> score 0.3510616719722748
Closest Category: <Wine> -> score 0.34688231348991394
Closest Category: <Retail> -> score 0.3149658441543579
